# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tessa-Saumu/FlyRank-ML-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> **Freestyle submission.** This notebook belongs to my final, freestyle capstone: *which ranking pages are losing clicks to AI-answer search before it shows, and which should an editor protect first?* My earlier Lane 2 (refresh) experiments are kept for the record in `work/archive/` and are not part of this submission.

**What this card asks:** a question, the unit of analysis, the output, the action, the cost of a wrong call, why data or ML
helps, and two or three real numbers. **The principle it serves:** a strong capstone starts with a useful question, not a
model. The question has to connect the data, a decision, an action and the cost of getting it wrong.

In [1]:
# Setup: works locally and in Colab. Installs only missing packages and never prints a token.
import importlib.util, json, os, sys
if any(importlib.util.find_spec(p) is None for p in ["duckdb", "huggingface_hub", "sklearn", "scipy", "matplotlib"]):
    get_ipython().run_line_magic("pip", "install -q duckdb huggingface_hub pandas scikit-learn scipy matplotlib")
here = os.path.abspath(os.getcwd())
ROOT = next((p for p in [here, os.path.dirname(here), os.path.dirname(os.path.dirname(here))]
             if os.path.isdir(os.path.join(p, "work", "scripts"))), None)
if ROOT is None:  # fresh Colab runtime: fetch the code (no data lives in git)
    get_ipython().system("git clone -q https://github.com/Tessa-Saumu/FlyRank-ML-Internship.git")
    ROOT = os.path.abspath("FlyRank-ML-Internship")
sys.path.insert(0, os.path.join(ROOT, "work", "scripts"))
OUT, FIG = os.path.join(ROOT, "work", "outputs"), os.path.join(ROOT, "work", "figures")
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings("ignore")
from IPython.display import Image, display, Markdown
import wh
DATA = wh.data_available(try_download=True)   # needs HF_TOKEN as an env var or Colab secret
print("pinned warehouse snapshot available:", DATA, "| revision", wh.REV[:8])
receipt = lambda name: json.load(open(os.path.join(OUT, name)))
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30)

pinned warehouse snapshot available: True | revision 50cbf7c3


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Question:** which lane, and why that one? **Why it matters (FlyRank principle: start from a useful question):** a lane
only helps if it connects data, a decision and an action.

I declare the **freestyle** track (lane guide, section 9: bring your own question; same bar as any lane).

I spent July to October on Lane 2 (content refresh: will a page's impressions fall by 20% next month?). The archive
shows what that taught me. My target was largely implied by momentum that was already visible: a two-line acceleration
rule (precision@50 of 0.812) beat every model. More features cannot rescue a target that the past already answers. So
the first requirement for this study was a target that recent momentum does **not** give away (see ML-03 and ML-05).

The second reason is FlyRank's own direction. AI answers in search (Google AI Overviews and AI Mode, ChatGPT,
Perplexity, Gemini) are the new variable in FlyRank's business. FlyRank's *State of AI-Driven SEO 2026* report
measures AI referral sessions, but it never measures what AI answers do to the clicks a page earns from Google at the
rank it already holds. That gap is a question the warehouse can answer.

In [2]:
mon = receipt("eda_ai_monthly.json")
print(f"Freestyle track on the full warehouse: {mon['daily_rows']:,} daily rows, {mon['first_date']} to {mon['last_date']}.")

Freestyle track on the full warehouse: 78,835,655 daily rows, 2025-01-27 to 2026-06-30.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Question:** which decision does this improve, who acts, and what does a wrong call cost? **Why it matters (FlyRank
principle: decision first):** an unactionable ranking is only a chart.

> **For** FlyRank's content team, **deciding** which ranking pages to work on before their clicks disappear, **I build**
> a monthly ranked queue of 50 top-10 pages from the daily Search Console warehouse, **scoring** the probability that a
> page earns at least 25% fewer clicks over the next two months than its current click-through rate and position
> predict. Success is **measured by** precision@50 against the base rate and against frozen rules, on months the model
> never saw. **A wrong call costs** an editor's review (false alarm) or a silent loss of clicks on a page that still
> ranks (miss). **A plain rule isn't enough because** the obvious rules ("top positions are most exposed", "a slipping
> CTR keeps slipping") point the wrong way at page level (ML-07). **I will claim only** observed, measured,
> decision-support results: Search Console has no AI-answer flag, so the pattern is *consistent with* AI answers
> absorbing clicks, not proof of it.

| Element | My answer |
|---|---|
| Unit of analysis | one page at one monthly decision point (page x origin month) |
| Output | a ranked queue of 50 pages, each with one reason code and one action |
| Who acts | a content editor, with the account lead for client-wide cases |
| Action | make the page citable in AI answers, rewrite for the click an AI answer cannot satisfy, hold, or escalate |
| Cost of a wrong call | false alarm: about 20 minutes of review; miss: clicks lost on a page that still ranks |
| Why data / ML | erosion depends on rank, recent CTR, demand, intent and the client together, and the mix shifts by month |

**How the question is answered, in three layers.** Each layer must earn its place before the next one:
1. **Does the effect exist?** Inferential statistics with the client as the unit of evidence (ML-06).
2. **Can a page-level target be defined honestly?** An observed label that momentum does not give away, with a measured
   noise floor (ML-03, ML-04, ML-05).
3. **Does a learned ranking beat simple rules out of time?** Forward validation with a sealed month (ML-07 to ML-09).
   Only then do I write a playbook (ML-10).

In [3]:
pb = receipt("erosion_playbook_summary.json")
print({k: pb[k] for k in ["queue_size", "client_cap", "reason_codes"]})

{'queue_size': 50, 'client_cap': 10, 'reason_codes': {'CTR_ABOVE_PEERS': 40, 'DEMAND_SHIFT': 10}}


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

**Why these three numbers:** each answers a doubt a reviewer should have. Is the problem real? Is it large? Why not
model AI referrals directly? All three come from the full warehouse (78.8 million daily rows), not the starter CSV.

In [4]:
e4, fe, mon = receipt("eda_04_client_level.json"), receipt("panel_fe.json"), receipt("eda_ai_monthly.json")
top10 = next(r for r in e4["A_client_level"] if r["segment"] == "1-10")
apr = next(r for r in fe["pos_1_3"]["effects"] if r["month"] == "2026-04")
ai_total = sum(r["ai"] or 0 for r in mon["months"])
print(f"1. Same page, same position: CTR fell in {top10['clients_down']} of {top10['clients']} clients between Oct-Nov 2025 and "
      f"Mar-Apr 2026 (median client CTR ratio {top10['median_client_ctr_ratio']}, Wilcoxon p = {top10['wilcoxon_p']}).")
print(f"2. Fixed-effects estimate, top-3 positions: April 2026 CTR = {apr['ctr_multiplier']} x October 2025 "
      f"(95% CI {apr['ci95'][0]} to {apr['ci95'][1]}, {fe['pos_1_3']['clients']} clients, {fe['pos_1_3']['pages']:,} pages).")
print(f"3. AI-assistant referral sessions in the whole warehouse: {ai_total:,.0f}, all from Oct 2025 onward: too thin to be the target.")

1. Same page, same position: CTR fell in 10 of 11 clients between Oct-Nov 2025 and Mar-Apr 2026 (median client CTR ratio 0.664, Wilcoxon p = 0.0068).
2. Fixed-effects estimate, top-3 positions: April 2026 CTR = 0.642 x October 2025 (95% CI 0.615 to 0.669, 35 clients, 19,000 pages).
3. AI-assistant referral sessions in the whole warehouse: 83,618, all from Oct 2025 onward: too thin to be the target.


**What these mean for the study.** The phenomenon is real and large (numbers 1 and 2), so a decision tool is worth
building. The direct AI-referral signal is too thin and too concentrated to be the target (number 3; the lane guide warns
against a classifier on AI sessions alone). So the target is the click loss in Google itself, where the data is rich,
and AI referrals enter only as a feature and as a finding (ML-06).

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**Question:** what may this work say, and what must it never say? **Why it matters (FlyRank principle: careful words):**
observational data supports decisions, not causal stories.

- **Observed:** at a fixed page and position, CTR fell after October 2025.
- **Measured:** the queue's precision against the base rate, on months the model never saw.
- **Directional:** transactional pages lost more CTR than informational pages in this panel.
- **Decision-support:** a ranked list that an editor reviews before acting.
- **Never claimed:** that AI answers caused the fall (Search Console has no AI-answer flag), that the model predicts
  Google, or that an action restores clicks (there is no hold-out yet).

In [5]:
sealed = receipt("erosion_model_metrics_sealed.json")["origins"]["2026-05"]
print(f"Observed: top-3 CTR in April 2026 = {apr['ctr_multiplier']} x October 2025 (95% CI {apr['ci95'][0]} to {apr['ci95'][1]}).")
print(f"Measured: sealed month P@50 = {sealed['metrics']['logreg']['p@50']} against a base rate of {sealed['base_rate']}.")

Observed: top-3 CTR in April 2026 = 0.642 x October 2025 (95% CI 0.615 to 0.669).
Measured: sealed month P@50 = 0.42 against a base rate of 0.2397.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.